# Extraction Demo

This notebook provides code to extract trial data from BDF recording files. 
It includes aggregating trial data, epoching, and extracting pre-trial baseline periods. Thereby, this script can be used as starting point of extracting, prepocessing, and analysing EEG data of your own experiments.

Data files for this script were recorded using the [example experiment](https://github.com/MaxHohmann/NeurospecMMBTS/blob/main/experiment_script.py) with varying sampling rate:
- `testdata_1024Hz_09112026.bdf`
- `testdata_2048Hz_09112026.bdf`
- `testdata_512Hz_09112026.bdf`
<br><br><br>

Additional information can be found on the **GitHub repository**: 
<br>
https://github.com/MaxHohmann/NeurospecMMBTS

## Setup

Import standard modules.

In [30]:
from pathlib import Path
import re

import mne
import numpy as np
import pandas as pd

mne.set_log_level('WARNING')

### Settings

Define directories and the trigger dictionary that links the numeric values of the **trigger line** to the corresponding **event names**. 

In [31]:
PROJECT_DIR = Path.cwd()                            # project directory
DATA_DIR    = Path(PROJECT_DIR, "example_data")     # data directory

print(f"Project directory   : {PROJECT_DIR}")
print(f"Data directory      : {DATA_DIR}")


# trigger values
TRIGGERS = {

    # trial
    "trial_start":  10,
    "trial_end":    11,

    "event_01":     20,
    "event_02":     21,
    "event_03":     22,
    "event_04":     23,
    "event_05":     24,

    # pause mode
    "PauseOff":     254,
    "PauseOn":      255,
}

Project directory   : /Users/maxhohmann/Desktop/NeurospecMMBTS
Data directory      : /Users/maxhohmann/Desktop/NeurospecMMBTS/example_data


### Select Recordings

Select all `.bdf` files within the **data folder**.

In [32]:
BDF_FILES = sorted(DATA_DIR.glob('*.bdf'))

print(f'{len(BDF_FILES)} BDF files:')

for file in BDF_FILES:
    print(f"\t- {file.name}")

3 BDF files:
	- testdata_1024Hz_09112026.bdf
	- testdata_2048Hz_09112026.bdf
	- testdata_512Hz_09112026.bdf


### Read BDF

MNE reads the BDF header and therefore uses the actual sampling frequency of each recording. Thus extraction works for recordings with various sampling frequencies alike.

In [36]:
raws = {}   # initiate dict

for file in BDF_FILES:
    raw = mne.io.read_raw_bdf(file, preload=False, verbose=False)
    raws[file.name] = raw

    print(f'\n{file.name}')
    print(f'    sfreq           : {raw.info["sfreq"]} Hz')
    print(f'    nchan           : {raw.info["nchan"]}')
    print(f'    duration        : {raw.times[-1]:.2f} s')
    print(f'    stim channels   : {[ch for ch in raw.ch_names if "status" in ch.lower() or "trigger" in ch.lower()]}')


testdata_1024Hz_09112026.bdf
    sfreq           : 1024.0 Hz
    nchan           : 73
    duration        : 38.00 s
    stim channels   : ['Status']

testdata_2048Hz_09112026.bdf
    sfreq           : 2048.0 Hz
    nchan           : 73
    duration        : 38.00 s
    stim channels   : ['Status']

testdata_512Hz_09112026.bdf
    sfreq           : 512.0 Hz
    nchan           : 73
    duration        : 38.00 s
    stim channels   : ['Status']


Get an overview of all channels of the recordings. Channels contain the **64** channels for **scalp electrodes**, additional **8** channels for **reference electrodes** (eye and mastoids), and **one** status channel for the **trigger line**.

In [ ]:
# get channel names
for i, ch in enumerate(raw.ch_names):
    print(i, ch)

0 Fp1
1 AF7
2 AF3
3 F1
4 F3
5 F5
6 F7
7 FT7
8 FC5
9 FC3
10 FC1
11 C1
12 C3
13 C5
14 T7
15 TP7
16 CP5
17 CP3
18 CP1
19 P1
20 P3
21 P5
22 P7
23 P9
24 PO7
25 PO3
26 O1
27 Iz
28 Oz
29 POz
30 Pz
31 CPz
32 Fpz
33 Fp2
34 AF8
35 AF4
36 AFz
37 Fz
38 F2
39 F4
40 F6
41 F8
42 FT8
43 FC6
44 FC4
45 FC2
46 FCz
47 Cz
48 C2
49 C4
50 C6
51 T8
52 TP8
53 CP6
54 CP4
55 CP2
56 P2
57 P4
58 P6
59 P8
60 P10
61 PO8
62 PO4
63 O2
64 M1
65 M2
66 IO1
67 LO2
68 LO1
69 EXG6
70 EXG7
71 EXG8
72 Status


### Extract Trial Events

Extract all relevant events from data files by reading the status channel directly and masking it to the lower 8 bits (the upper bits of the BioSemi status channel carry other flags, e.g. battery/CMS status, and are not part of the trigger code).

In [ ]:
def extract_events(raw, triggers):
    """
    Extract all relevant trigger events from the status channel.
    """

    # find status channel
    stim_ch = [ch for ch in raw.ch_names if 'Status' in ch]
    if not stim_ch:
        raise RuntimeError('No status channel found!')

    # read status channel and convert to integer
    status = raw.get_data(picks=stim_ch)[0]
    status = np.round(status).astype(np.int64)

    # keep only the lower 8 bits (actual trigger code)
    status = status & 0xFF

    # detect changes (event onsets)
    changes = np.diff(status, prepend=status[0])
    onset_samples = np.where(changes != 0)[0]
    onset_samples = onset_samples[onset_samples > 0]


    # # get events
    # events = np.column_stack([
    #     onset_samples,
    #     status[onset_samples - 1],
    #     status[onset_samples],
    # ])

    # get events
    events = mne.find_events(
        raw,
        stim_channel    = stim_ch[0],
        consecutive     = True,
        shortest_event  = 1,
        output          = 'onset',
        verbose         = False
    )


    # keep only defined trigger values
    trigger_names = {value: name for name, value in triggers.items()}
    events = events[np.isin(events[:, 2], list(trigger_names.keys()))]

    # create data frame
    events_df = pd.DataFrame({
        'sample':         events[:, 0].astype(int),
        'previous_value': events[:, 1].astype(int),
        'trigger':        events[:, 2].astype(int),
    })

    # add sample time in s
    events_df['time_s'] = events_df['sample'] / raw.info['sfreq']

    # add trigger name
    events_df['name'] = events_df['trigger'].map(trigger_names)


    return events_df

In [39]:
event_tables = {}

for name, raw in raws.items():
    event_tables[name] = extract_events(raw, TRIGGERS)

    print(f'\n{name}')
    display(event_tables[name])


testdata_1024Hz_09112026.bdf


,sample,previous_value,trigger,time_s,name
0,772,0,10,0.753906,trial_start
1,1806,0,20,1.763672,event_01
2,6942,0,11,6.779297,trial_end
3,8505,0,10,8.305664,trial_start
4,9540,0,21,9.316406,event_02
5,14661,0,11,14.317383,trial_end
6,15712,65790,254,15.343750,PauseOff
7,16226,0,10,15.845703,trial_start
8,17259,0,22,16.854492,event_03
9,22395,0,11,21.870117,trial_end



testdata_2048Hz_09112026.bdf


,sample,previous_value,trigger,time_s,name
0,2624,0,10,1.281250,trial_start
1,4697,0,20,2.293457,event_01
2,14953,0,11,7.301270,trial_end
3,18056,0,10,8.816406,trial_start
4,20124,0,21,9.826172,event_02
5,30376,0,11,14.832031,trial_end
6,33497,0,10,16.355957,trial_start
7,35570,0,22,17.368164,event_03
8,45840,0,11,22.382812,trial_end
9,48980,0,10,23.916016,trial_start



testdata_512Hz_09112026.bdf


,sample,previous_value,trigger,time_s,name
0,775,0,10,1.513672,trial_start
1,1291,0,20,2.521484,event_01
2,3856,0,11,7.531250,trial_end
3,4639,0,10,9.060547,trial_start
4,5157,0,21,10.072266,event_02
5,7724,0,11,15.085938,trial_end
6,8248,65790,254,16.109375,PauseOff
7,8507,0,10,16.615234,trial_start
8,9022,0,22,17.621094,event_03
9,11587,0,11,22.630859,trial_end


### Build trial table

Trial boundaries are defined **from one `stream_start` (254) to the next `stream_start`** (or the end of the recording for the last trial). This is more robust than relying on `stream_stop`/`trial_end`, because:
- `stream_stop` (255) is reserved by the recording software (see note above) and is essentially never logged.
- `trial_end` (11) can be missed too, especially with a short pause before the next `stream_start` (see the diagnostic section at the end of this notebook).

Within each `stream_start`–`stream_start` block we look for:
- `trial_start` (10) → marks the end of the **pre-trial rest phase** and the start of the actual trial,
- the **event trigger** (20-24) → the stimulus/condition of that trial,
- `trial_end` (11), if present, purely informational.

This gives us, per trial, both the **pre-trial rest phase** (`stream_start` → `trial_start`) for baseline purposes, and the **trial phase** (`trial_start` → end of block) for the actual analysis.

In [40]:
def build_trial_table(events_df, triggers, recording_name, n_samples, sfreq):
    """
    Create one row per trial from a recording's event table.
    Each trial spans from one 'PauseOff' trigger to the next 'PauseOff'
    (or the end of the recording for the last trial in a file).
    """

    stream_start_code = triggers['PauseOff']
    trial_start_code = triggers['trial_start']
    trial_end_code = triggers.get('trial_end')
    event_codes = [v for k, v in triggers.items() if k.startswith('event')]

    starts = events_df[events_df['trigger'] == stream_start_code].reset_index(drop=True)

    rows = []
    for i, srow in starts.iterrows():
        block_start_sample = srow['sample']
        block_start_time = srow['time_s']

        if i + 1 < len(starts):
            block_end_sample = int(starts.loc[i + 1, 'sample'])
            block_end_time = starts.loc[i + 1, 'time_s']
        else:
            block_end_sample = n_samples - 1
            block_end_time = block_end_sample / sfreq

        within = events_df[
            (events_df['sample'] >= block_start_sample) &
            (events_df['sample'] < block_end_sample)
        ]

        ts_rows = within[within['trigger'] == trial_start_code]
        trial_start = ts_rows.iloc[0] if len(ts_rows) else None

        ev_rows = within[within['trigger'].isin(event_codes)]
        event = ev_rows.iloc[0] if len(ev_rows) else None

        te_rows = within[within['trigger'] == trial_end_code] if trial_end_code else within.iloc[0:0]
        trial_end = te_rows.iloc[0] if len(te_rows) else None

        rows.append({
            'recording':            recording_name,
            'trial':                i + 1,
            'PauseOff_sample':  int(block_start_sample),
            'PauseOff_time_s':  block_start_time,
            'trial_start_sample':   int(trial_start['sample']) if trial_start is not None else np.nan,
            'trial_start_time_s':   trial_start['time_s'] if trial_start is not None else np.nan,
            'event_trigger':        int(event['trigger']) if event is not None else np.nan,
            'event_name':           event['name'] if event is not None else None,
            'event_sample':         int(event['sample']) if event is not None else np.nan,
            'event_time_s':         event['time_s'] if event is not None else np.nan,
            'trial_end_recorded':   trial_end is not None,
            'trial_end_sample':     int(trial_end['sample']) if trial_end is not None else np.nan,
            'trial_end_time_s':     trial_end['time_s'] if trial_end is not None else np.nan,
            'block_end_sample':     block_end_sample,
            'block_end_time_s':     block_end_time,
            'pre_trial_duration_s': (trial_start['time_s'] - block_start_time) if trial_start is not None else np.nan,
            'event_latency_s':      (event['time_s'] - trial_start['time_s']) if (trial_start is not None and event is not None) else np.nan,
            'block_duration_s':     block_end_time - block_start_time,
        })

    return pd.DataFrame(rows)

In [41]:
trial_tables = {}   # one DataFrame per recording (needed later for epoching)
trial_list = []      # for the combined/aggregated table

for name, raw in raws.items():
    tdf = build_trial_table(event_tables[name], TRIGGERS, name, len(raw.times), raw.info['sfreq'])
    trial_tables[name] = tdf
    trial_list.append(tdf)

trials_all = pd.concat(trial_list, ignore_index=True)

n_missing_end = (~trials_all['trial_end_recorded']).sum()
print(f'Trials without a recorded trial_end trigger: {n_missing_end}/{len(trials_all)} '
      f'(harmless here, trial boundaries do not rely on it)')

display(trials_all)

Trials without a recorded trial_end trigger: 0/3 (harmless here, trial boundaries do not rely on it)


,recording,trial,PauseOff_sample,PauseOff_time_s,trial_start_sample,trial_start_time_s,event_trigger,event_name,event_sample,event_time_s,trial_end_recorded,trial_end_sample,trial_end_time_s,block_end_sample,block_end_time_s,pre_trial_duration_s,event_latency_s,block_duration_s
0,testdata_1024Hz_09112026.bdf,1,15712,15.343750,16226,15.845703,22,event_03,17259,16.854492,True,22395,21.870117,38911,37.999023,0.501953,1.008789,22.655273
1,testdata_512Hz_09112026.bdf,1,8248,16.109375,8507,16.615234,22,event_03,9022,17.621094,True,11587,22.630859,15984,31.218750,0.505859,1.005859,15.109375
2,testdata_512Hz_09112026.bdf,2,15984,31.218750,16240,31.718750,24,event_05,16756,32.726562,True,19320,37.734375,19455,37.998047,0.500000,1.007812,6.779297


### Save aggregated trial table

The resulting CSV contains one row per trial (across all recordings) and keeps both sample indices and seconds. Keeping both is useful because sample indices depend on the recording's sampling rate, whereas seconds allow direct comparison between recordings at different sampling rates.

In [ ]:
OUT = DATA_DIR / 'trials_aggregated.csv'
trials_all.to_csv(OUT, index=False)
print(f'Saved: {OUT.resolve()}')

Saved: /Users/maxhohmann/Desktop/NeurospecMMBTS/example_data/trials_aggregated.csv


## Build MNE Epochs (pre-trial baseline + trial phase)

For EEG preprocessing we usually want the actual EEG signal per trial, not just the timing table. We build one `Epochs` object per recording:

- **t = 0** is set to `trial_start`.
- **`tmin`** reaches back far enough to cover the pre-trial rest phase (`stream_start` → `trial_start`) for every trial.
- **`tmax`** reaches forward to the end of the shortest available trial block, so every epoch has valid data.
- **`baseline=(None, 0)`** applies baseline correction using exactly that pre-trial rest phase.
- Trial metadata (recording name, event name, latencies, whether `trial_end` was recorded, …) is attached via `epochs.metadata`.

`tmin`/`tmax` are computed **globally** across all recordings currently in `DATA_DIR` (not per file), so the resulting epochs are directly comparable in time and can later be resampled to a common sampling rate and combined.

In [ ]:
MARGIN = 0.02   # safety margin (s) to avoid rounding overruns at the data edges

pre_durations = trials_all['pre_trial_duration_s'].dropna()
post_durations = (trials_all['block_end_time_s'] - trials_all['trial_start_time_s']).dropna()

GLOBAL_TMIN = -(pre_durations.min() - MARGIN)
GLOBAL_TMAX = post_durations.min() - MARGIN

print(f'GLOBAL_TMIN = {GLOBAL_TMIN:.3f} s   (shortest pre-trial rest phase across all trials, minus margin)')
print(f'GLOBAL_TMAX = {GLOBAL_TMAX:.3f} s   (shortest trial block across all trials, minus margin)')

GLOBAL_TMIN = -0.484 s   (shortest pre-trial rest phase across all trials, minus margin)
GLOBAL_TMAX = 5.528 s   (shortest trial block across all trials, minus margin)


In [ ]:
def build_epochs(raw, trial_df, triggers, tmin, tmax):
    """Build one Epochs object for a recording, time-locked to 'trial_start'."""

    valid = trial_df.dropna(subset=['trial_start_sample', 'event_trigger']).copy()
    n_dropped = len(trial_df) - len(valid)
    if n_dropped:
        print(f'  skipping {n_dropped} trial(s) without trial_start/event trigger')
    if valid.empty:
        return None

    events_arr = np.column_stack([
        valid['trial_start_sample'].astype(int).values,
        np.zeros(len(valid), dtype=int),
        valid['event_trigger'].astype(int).values,
    ])

    present_codes = set(valid['event_trigger'].astype(int))
    event_id = {k: v for k, v in triggers.items() if k.startswith('event') and v in present_codes}

    epochs = mne.Epochs(
        raw, events_arr, event_id=event_id,
        tmin=tmin, tmax=tmax,
        baseline=(None, 0),      # baseline-correct using the pre-trial rest phase
        preload=True, verbose=False,
    )
    epochs.metadata = valid.reset_index(drop=True)

    return epochs

In [ ]:
epochs_all = {}

for name, raw in raws.items():
    print(name)
    epochs = build_epochs(raw, trial_tables[name], TRIGGERS, GLOBAL_TMIN, GLOBAL_TMAX)
    if epochs is not None:
        print(f'  n_epochs={len(epochs)}  tmin={epochs.tmin:.3f}s  tmax={epochs.tmax:.3f}s  '
              f'sfreq={raw.info["sfreq"]}Hz  n_times={len(epochs.times)}')
        epochs_all[name] = epochs

new_DTR_True_1000ms_1024Hz_25092026.bdf
  n_epochs=5  tmin=-0.484s  tmax=5.528s  sfreq=1024.0Hz  n_times=6158
new_DTR_True_1000ms_2048Hz_25092026.bdf
  n_epochs=5  tmin=-0.484s  tmax=5.528s  sfreq=2048.0Hz  n_times=12313
new_DTR_True_1000ms_512Hz_25092026.bdf
  n_epochs=5  tmin=-0.484s  tmax=5.527s  sfreq=512.0Hz  n_times=3079
new_DTR_True_1024Hz_25092026.bdf
  n_epochs=5  tmin=-0.484s  tmax=5.528s  sfreq=1024.0Hz  n_times=6158
new_DTR_True_2048Hz_25092026.bdf
  n_epochs=5  tmin=-0.484s  tmax=5.528s  sfreq=2048.0Hz  n_times=12313
testdata_1000ms_1024Hz_09112026.bdf
  n_epochs=5  tmin=-0.484s  tmax=5.528s  sfreq=1024.0Hz  n_times=6158
testdata_1000ms_2048Hz_09112026.bdf
  n_epochs=5  tmin=-0.484s  tmax=5.528s  sfreq=2048.0Hz  n_times=12313
testdata_1000ms_512Hz_09112026.bdf
  n_epochs=5  tmin=-0.484s  tmax=5.527s  sfreq=512.0Hz  n_times=3079


### Save Epochs

Save one `-epo.fif` file per recording, so they can be loaded directly in later preprocessing/analysis scripts without re-running the extraction.

In [ ]:
for name, epochs in epochs_all.items():
    out_path = DATA_DIR / f'{Path(name).stem}-epo.fif'
    epochs.save(out_path, overwrite=True)
    print(f'Saved: {out_path}')

Saved: /Users/maxhohmann/Desktop/NeurospecMMBTS/example_data/new_DTR_True_1000ms_1024Hz_25092026-epo.fif
Saved: /Users/maxhohmann/Desktop/NeurospecMMBTS/example_data/new_DTR_True_1000ms_2048Hz_25092026-epo.fif
Saved: /Users/maxhohmann/Desktop/NeurospecMMBTS/example_data/new_DTR_True_1000ms_512Hz_25092026-epo.fif
Saved: /Users/maxhohmann/Desktop/NeurospecMMBTS/example_data/new_DTR_True_1024Hz_25092026-epo.fif
Saved: /Users/maxhohmann/Desktop/NeurospecMMBTS/example_data/new_DTR_True_2048Hz_25092026-epo.fif
Saved: /Users/maxhohmann/Desktop/NeurospecMMBTS/example_data/testdata_1000ms_1024Hz_09112026-epo.fif
Saved: /Users/maxhohmann/Desktop/NeurospecMMBTS/example_data/testdata_1000ms_2048Hz_09112026-epo.fif
Saved: /Users/maxhohmann/Desktop/NeurospecMMBTS/example_data/testdata_1000ms_512Hz_09112026-epo.fif


### Optional: combine recordings across sampling rates

Since `tmin`/`tmax` were computed globally, the epochs from all recordings cover the same time window and can be resampled to a common sampling rate and concatenated into a single `Epochs` object. Resampling different original rates to a common one can leave a one-sample rounding difference, so we trim all epochs to the shortest common length before concatenating.

In [ ]:
sfreq_common = min(raw.info['sfreq'] for raw in raws.values())
print(f'Resampling all recordings to {sfreq_common} Hz...')

resampled = []
for name, epochs in epochs_all.items():
    ep = epochs.copy().resample(sfreq_common)
    resampled.append(ep)

# Trim to the shortest common number of time points (rounding safety), and
# force one common tmin (GLOBAL_TMIN) instead of each epoch's own ep.tmin:
# depending on the *original* sampling rate, ep.tmin can round to a slightly
# different float even though it targets the same GLOBAL_TMIN, which can make
# mne recompute one extra/one fewer time point than the trimmed data actually
# has. Using the same explicit tmin for every EpochsArray avoids that mismatch.
min_len = min(len(ep.times) for ep in resampled)
trimmed = []
for ep in resampled:
    data = ep.get_data(copy=True)[:, :, :min_len]
    ea = mne.EpochsArray(
        data, ep.info, events=ep.events, tmin=GLOBAL_TMIN,
        event_id=ep.event_id, metadata=ep.metadata, verbose=False,
    )
    trimmed.append(ea)

epochs_combined = mne.concatenate_epochs(trimmed)
print(epochs_combined)

OUT_COMBINED = DATA_DIR / 'trials_combined-epo.fif'
epochs_combined.save(OUT_COMBINED, overwrite=True)
print(f'Saved: {OUT_COMBINED.resolve()}')

Resampling all recordings to 512.0 Hz...
<EpochsArray | 40 events (all good), -0.484 – 5.525 s (baseline off), ~68.6 MiB, data loaded, with metadata,
 'event_01': 8
 'event_02': 8
 'event_03': 8
 'event_04': 8
 'event_05': 8>
Saved: /Users/maxhohmann/Desktop/NeurospecMMBTS/example_data/trials_combined-epo.fif
